In [ ]:
import pandas as pd

# Define the relative path to the CSV file
metadata_path = '../COVID-US/utils/LUSS_metadata.csv'

# Read the CSV file using the relative path
metadata = pd.read_csv(metadata_path)

# Display the data 
print(metadata.head())


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def parse_filename(filename):
    parts = filename.split('_')
    video_number = int(parts[0])
    return video_number

# Example usage
filename = "1_butterfly_covid_prc_convex_clean_frame0.jpg"
video_number = parse_filename(filename)
print(video_number)  # Output: 1


In [ ]:
RESIZED_IMAGE= '../COVID-US/data/image/clean/resized'

def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# Example usage
resized_dir = '../COVID-US/data/image/clean/resized'
ensure_directory_exists(RESIZED_IMAGE)


In [ ]:
def preprocess_and_save_image(image_path, resized_dir, target_size=(224, 224), normalization_range=(0, 1)):
    filename = os.path.basename(image_path)
    resized_filename = filename.replace('.jpg', '_resized.jpg')
    resized_image_path = os.path.join(resized_dir, resized_filename)
    
    # Check if the resized image already exists
    if os.path.exists(resized_image_path):
        image = cv2.imread(resized_image_path)
        if image is not None:
            return image
    
    # If the resized image does not exist, create it
    image = cv2.imread(image_path)
    if image is not None:
        image = cv2.resize(image, target_size)
        cv2.imwrite(resized_image_path, image)  # Save resized image
        
        # Normalize image
        image = image.astype('float32')
        if normalization_range == (0, 1):
            image /= 255.0
        elif normalization_range == (-1, 1):
            image = (image / 127.5) - 1.0
        return image
    return None


In [ ]:
def load_and_preprocess_images(image_dir, resized_dir, metadata, target_size=(224, 224), normalization_range=(0, 1)):
    image_data = []
    labels = []
    probe_types = []
    image_paths = []
    
    ensure_directory_exists(resized_dir)
    
    for filename in os.listdir(image_dir):
        if filename.endswith('.jpg'):
            video_number = parse_filename(filename)
            img_path = os.path.join(image_dir, filename)
            image = preprocess_and_save_image(img_path, resized_dir, target_size, normalization_range)
            if image is not None:
                image_data.append(image)
                image_paths.append(img_path)
                
                # Extract corresponding metadata
                video_metadata = metadata[metadata['id'].str.startswith(str(video_number) + '_')]
                if not video_metadata.empty:
                    labels.append(video_metadata['class'].values[0])
                    probe_types.append(video_metadata['probe'].values[0])
                else:
                    print(f"No metadata found for video number {video_number}")

    return np.array(image_data), labels, probe_types, image_paths

CLEAN_IMAGE = '../COVID-US/data/image/clean'
RESIZED_IMAGE = '../COVID-US/data/image/clean/resized'
images, labels, probe_types, image_paths = load_and_preprocess_images(CLEAN_IMAGE, RESIZED_IMAGE, metadata)


In [ ]:
# print(probe_types)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

# Extract video IDs from image paths (assuming filenames are formatted as "videoID_frameID_resized.jpg")
video_ids = [os.path.basename(path).split('_')[0] for path in image_paths]

# Convert to NumPy array
video_ids = np.array(video_ids)

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# One-hot encode probe types
probe_type_encoder = OneHotEncoder(sparse_output=False)
encoded_probe_types = probe_type_encoder.fit_transform(np.array(probe_types).reshape(-1, 1))

# Create a DataFrame with minimal information for splitting
data = pd.DataFrame({
    'video_id': video_ids,
    'label': encoded_labels
})

# Map video IDs to their corresponding labels
video_to_label = data.groupby('video_id')['label'].apply(lambda x: x.mode()[0]).to_dict()

# Get the label for each unique video ID
unique_video_ids = data['video_id'].unique()
video_labels = np.array([video_to_label[vid] for vid in unique_video_ids])

# Split video IDs into training, validation, and test sets
train_videos, test_videos = train_test_split(unique_video_ids, test_size=0.1, random_state=42, stratify=video_labels)
train_labels = np.array([video_to_label[vid] for vid in train_videos])
train_videos, val_videos = train_test_split(train_videos, test_size=0.2, random_state=42, stratify=train_labels)

# Assign frames to the respective sets based on their video split
train_indices = data[data['video_id'].isin(train_videos)].index
val_indices = data[data['video_id'].isin(val_videos)].index
test_indices = data[data['video_id'].isin(test_videos)].index

# Use indices to extract the actual data
X_train = np.stack([images[i] for i in train_indices])
y_train = encoded_labels[train_indices]
probe_train = np.stack([encoded_probe_types[i] for i in train_indices])

X_val = np.stack([images[i] for i in val_indices])
y_val = encoded_labels[val_indices]
probe_val = np.stack([encoded_probe_types[i] for i in val_indices])

X_test = np.stack([images[i] for i in test_indices])
y_test = encoded_labels[test_indices]
probe_test = np.stack([encoded_probe_types[i] for i in test_indices])

# Validate input shapes
print("X_train shape:", X_train.shape)
print("probe_train shape:", probe_train.shape)
print("X_val shape:", X_val.shape)
print("probe_val shape:", probe_val.shape)
print("X_test shape:", X_test.shape)
print("probe_test shape:", probe_test.shape)


In [ ]:
# Record the distribution of labels in each set
train_label_distribution = pd.Series(y_train).value_counts().sort_index()
val_label_distribution = pd.Series(y_val).value_counts().sort_index()
test_label_distribution = pd.Series(y_test).value_counts().sort_index()

# Map encoded labels back to original class names
label_class_names = label_encoder.inverse_transform(train_label_distribution.index)

# Create a DataFrame to store the distributions with class names
label_distributions = pd.DataFrame({
    'Class Name': label_class_names,
    'Train': train_label_distribution.values,
    'Validation': val_label_distribution.values,
    'Test': test_label_distribution.values
}).set_index('Class Name')

# Display the DataFrame
print(label_distributions)


In [ ]:
# Plot the label distributions
plt.figure(figsize=(12, 6))
label_distributions.plot(kind='bar')
plt.title('Label Distribution')
plt.xlabel('Class Name')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='best')
plt.show()


In [ ]:
import os
import certifi
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.applications import (
    MobileNetV2, InceptionV3, InceptionResNetV2, Xception, ResNet50V2,
    EfficientNetB4, DenseNet121, DenseNet169, DenseNet201
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def build_baseline_model(base_model, input_shape=(224, 224, 3), num_classes=4, fine_tune=True):
    # Load the base model with pre-trained ImageNet weights
    base_model = base_model(include_top=False, input_shape=input_shape, weights='imagenet')
    
    # Freeze the convolutional layers during the first phase of training
    if not fine_tune:
        for layer in base_model.layers:
            layer.trainable = False
    else:
        # Unfreeze all layers for fine-tuning
        for layer in base_model.layers:
            layer.trainable = True
    
    cnn_input = Input(shape=input_shape, name='cnn_input')
    cnn_output = base_model(cnn_input)
    cnn_output = Flatten()(cnn_output)
    final_output = Dense(num_classes, activation='softmax')(cnn_output)

    model = Model(inputs=cnn_input, outputs=final_output)
    
    # Use a low learning rate when fine-tuning the model
    learning_rate = 0.0001 if not fine_tune else 0.00001
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model


def build_multimodal_model(base_model, input_shape=(224, 224, 3), num_classes=4, fine_tune=True):
    base_model = base_model(include_top=False, input_shape=input_shape, weights='imagenet')
    
    if not fine_tune:
        for layer in base_model.layers:
            layer.trainable = False
    else:
        # Unfreeze all layers for fine-tuning
        for layer in base_model.layers:
            layer.trainable = True
    
    cnn_input = Input(shape=input_shape, name='cnn_input')
    cnn_output = base_model(cnn_input)
    cnn_output = Flatten()(cnn_output)

    probe_input_shape = (encoded_probe_types.shape[1],)
    probe_input = Input(shape=probe_input_shape, name='probe_input')
    x = Dense(1024, activation='relu')(probe_input)
    x = Dropout(0.4)(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(512, activation='relu')(x)

    combined = Concatenate()([cnn_output, x])
    final_output = Dense(num_classes, activation='softmax')(combined)

    model = Model(inputs=[cnn_input, probe_input], outputs=final_output)
    
    # Use a low learning rate when fine-tuning the model
    learning_rate = 0.0001 if not fine_tune else 0.00001
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model



In [ ]:
def train_model(model, X_train, y_train, probe_train, X_val, y_val, probe_val, epochs=20, batch_size=16, multimodal=True):
    callbacks = [
        # EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=2, min_lr=1e-6)
    ]
    
    if multimodal:
        history = model.fit(
            [X_train, probe_train], y_train,
            validation_data=([X_val, probe_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks
        )
    else:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks
        )
    return history

from sklearn.metrics import confusion_matrix, accuracy_score

def evaluate_model(model, X_test, y_test, probe_test, multimodal=True):
    if multimodal:
        y_pred = model.predict([X_test, probe_test])
    else:
        y_pred = model.predict(X_test)
    
    y_pred_classes = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_test, y_pred_classes)
    conf_matrix = confusion_matrix(y_test, y_pred_classes)
    
    print(f'Accuracy: {accuracy * 100:.2f}%')
    print('Confusion Matrix:')
    print(conf_matrix)
    return accuracy, conf_matrix



In [ ]:
# Validate input shapes
print("X_train shape:", X_train.shape)
print("probe_train shape:", probe_train.shape)
print("X_val shape:", X_val.shape)
print("probe_val shape:", probe_val.shape)
print("X_test shape:", X_test.shape)
print("probe_test shape:", probe_test.shape)


In [ ]:
# # Step 1: Modify `run_model_evaluation` to return metrics
# def run_model_evaluation(base_model_name, base_model, X_train, y_train, probe_train, X_val, y_val, probe_val, X_test, y_test, probe_test):
#     metrics = {
#         'model_name': base_model_name,
#         'baseline': {},
#         'multimodal': {}
#     }
    
#     print(f'Evaluating {base_model_name} Baseline Model')
#     baseline_model = build_baseline_model(base_model)
#     baseline_history = train_model(baseline_model, X_train, y_train, None, X_val, y_val, None, multimodal=False)
#     baseline_accuracy, baseline_conf_matrix = evaluate_model(baseline_model, X_test, y_test, None, multimodal=False)
    
#     metrics['baseline']['history'] = baseline_history.history
#     metrics['baseline']['accuracy'] = baseline_accuracy
#     metrics['baseline']['conf_matrix'] = baseline_conf_matrix
    
#     print(f'Evaluating {base_model_name} Multimodal Model')
#     multimodal_model = build_multimodal_model(base_model)
#     multimodal_history = train_model(multimodal_model, X_train, y_train, probe_train, X_val, y_val, probe_val, multimodal=True)
#     multimodal_accuracy, multimodal_conf_matrix = evaluate_model(multimodal_model, X_test, y_test, probe_test, multimodal=True)
    
#     metrics['multimodal']['history'] = multimodal_history.history
#     metrics['multimodal']['accuracy'] = multimodal_accuracy
#     metrics['multimodal']['conf_matrix'] = multimodal_conf_matrix
    
#     return metrics


In [ ]:
def run_model_evaluation(base_model_name, base_model, X_train, y_train, probe_train, X_val, y_val, probe_val, X_test, y_test, probe_test, results, epochs=20):
    metrics = {
        'model_name': base_model_name,
        'baseline': {},
        'multimodal': {}
    }
    
    # Define paths for model weights
    baseline_weights_path = f'./model_weights/{base_model_name}_baseline.weights.h5'
    multimodal_weights_path = f'./model_weights/{base_model_name}_multimodal.weights.h5'
    
    # Check if the model results already exist in the results list
    existing_model_metrics = next((item for item in results if item['model_name'] == base_model_name), None)
    
    retrain_baseline = True
    retrain_multimodal = True

    # Check conditions for skipping baseline training
    if existing_model_metrics:
        baseline_history_len = len(existing_model_metrics['baseline'].get('history', {}).get('accuracy', []))
        if os.path.exists(baseline_weights_path) and baseline_history_len == epochs:
            retrain_baseline = False

    # Check conditions for skipping multimodal training
    if existing_model_metrics:
        multimodal_history_len = len(existing_model_metrics['multimodal'].get('history', {}).get('accuracy', []))
        if os.path.exists(multimodal_weights_path) and multimodal_history_len == epochs:
            retrain_multimodal = False

    # Evaluate baseline model
    print(f'Evaluating {base_model_name} Baseline Model')
    baseline_model = build_baseline_model(base_model)
    
    if retrain_baseline:
        baseline_history = train_model(baseline_model, X_train, y_train, None, X_val, y_val, None, epochs=epochs, multimodal=False)
        baseline_model.save_weights(baseline_weights_path)
        metrics['baseline']['history'] = baseline_history.history
    else:
        print(f"Loading existing weights for {base_model_name} baseline model.")
        baseline_model.load_weights(baseline_weights_path)
        metrics['baseline'] = existing_model_metrics['baseline']

    baseline_accuracy, baseline_conf_matrix = evaluate_model(baseline_model, X_test, y_test, None, multimodal=False)
    metrics['baseline']['accuracy'] = baseline_accuracy
    metrics['baseline']['conf_matrix'] = baseline_conf_matrix
    
    # Evaluate multimodal model
    print(f'Evaluating {base_model_name} Multimodal Model')
    multimodal_model = build_multimodal_model(base_model)
    
    if retrain_multimodal:
        multimodal_history = train_model(multimodal_model, X_train, y_train, probe_train, X_val, y_val, probe_val, epochs=epochs, multimodal=True)
        multimodal_model.save_weights(multimodal_weights_path)
        metrics['multimodal']['history'] = multimodal_history.history
    else:
        print(f"Loading existing weights for {base_model_name} multimodal model.")
        multimodal_model.load_weights(multimodal_weights_path)
        metrics['multimodal'] = existing_model_metrics['multimodal']

    multimodal_accuracy, multimodal_conf_matrix = evaluate_model(multimodal_model, X_test, y_test, probe_test, multimodal=True)
    metrics['multimodal']['accuracy'] = multimodal_accuracy
    metrics['multimodal']['conf_matrix'] = multimodal_conf_matrix
    
    return metrics


In [ ]:
import os
import pickle

# Ensure the directory for storing model weights exists
os.makedirs('./model_weights', exist_ok=True)

# Check if results.pkl exists
if os.path.exists('results.pkl'):
    with open('results.pkl', 'rb') as file:
        results = pickle.load(file)
else: results = []

models = { 
    "MobileNetV2": MobileNetV2,
    "InceptionV3": InceptionV3,
    "InceptionResNetV2": InceptionResNetV2,
    "Xception": Xception,
    "ResNet50V2": ResNet50V2,
    "EfficientNetB4": EfficientNetB4,
    "DenseNet121": DenseNet121,
    "DenseNet169": DenseNet169,
    "DenseNet201": DenseNet201
}

# Store results in a list
for model_name, model_class in models.items():
    metrics = run_model_evaluation(
        model_name, 
        model_class, 
        X_train, y_train, probe_train, 
        X_val, y_val, probe_val, 
        X_test, y_test, probe_test, 
        results
    )
    # Update the results list, ensuring no duplicates
    results = [item for item in results if item['model_name'] != model_name]
    results.append(metrics)

# Save the results to a file
with open('results.pkl', 'wb') as file:
    pickle.dump(results, file)

# Step 3: Convert results to DataFrame
import pandas as pd

results_df = pd.DataFrame(results)

accuracy_data = []
for result in results:
    accuracy_data.append({
        'model_name': result['model_name'],
        'model_type': 'baseline',
        'accuracy': result['baseline']['accuracy']
    })
    accuracy_data.append({
        'model_name': result['model_name'],
        'model_type': 'multimodal',
        'accuracy': result['multimodal']['accuracy']
    })

accuracy_df = pd.DataFrame(accuracy_data)


In [ ]:
print(accuracy_df)

In [ ]:
print(results)

In [ ]:
def print_model_summaries(base_model_name, base_model_class):
    # Define paths for model weights
    baseline_weights_path = f'./model_weights/{base_model_name}_baseline.weights.h5'
    multimodal_weights_path = f'./model_weights/{base_model_name}_multimodal.weights.h5'
    
    # Build and load the baseline model
    print(f"Building and loading the baseline model for {base_model_name}...")
    baseline_model = build_baseline_model(base_model_class)
    if os.path.exists(baseline_weights_path):
        baseline_model.load_weights(baseline_weights_path)
        print(f"Loaded weights from {baseline_weights_path}")
    else:
        print(f"Weights file for baseline model not found at {baseline_weights_path}")
    
    # Build and load the multimodal model
    print(f"Building and loading the multimodal model for {base_model_name}...")
    multimodal_model = build_multimodal_model(base_model_class)
    if os.path.exists(multimodal_weights_path):
        multimodal_model.load_weights(multimodal_weights_path)
        print(f"Loaded weights from {multimodal_weights_path}")
    else:
        print(f"Weights file for multimodal model not found at {multimodal_weights_path}")
    
    # Print summaries
    print(f"\nSummary for Baseline Model ({base_model_name}):")
    baseline_model.summary()
    
    print(f"\nSummary for Multimodal Model ({base_model_name}):")
    multimodal_model.summary()

print_model_summaries("MobileNetV2",MobileNetV2)

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = np.mean(y_true == y_pred) * 100
    precision = precision_score(y_true, y_pred, average='weighted') * 100
    recall = recall_score(y_true, y_pred, average='weighted') * 100
    f1 = f1_score(y_true, y_pred, average='weighted') * 100
    return accuracy, precision, recall, f1

In [ ]:
# def evaluate_trained_models(models, X_test, y_test, probe_test, results):
#     evaluation_data = []
    
#     for model_name, model_class in models.items():
#         # Define paths for model weights
#         baseline_weights_path = f'./model_weights/{model_name}_baseline.weights.h5'
#         multimodal_weights_path = f'./model_weights/{model_name}_multimodal.weights.h5'
        
#         # Evaluate baseline model
#         print(f'Evaluating {model_name} Baseline Model')
#         baseline_model = build_baseline_model(model_class)
#         if os.path.exists(baseline_weights_path):
#             baseline_model.load_weights(baseline_weights_path)
#             y_pred_baseline = np.argmax(baseline_model.predict(X_test), axis=1)
#             accuracy_baseline, precision_baseline, recall_baseline, f1_baseline = calculate_metrics(y_test, y_pred_baseline)
#             params_baseline = baseline_model.count_params()
#         else:
#             print(f"Weights for baseline model not found at {baseline_weights_path}")
#             continue
        
#         # Evaluate multimodal model
#         print(f'Evaluating {model_name} Multimodal Model')
#         multimodal_model = build_multimodal_model(model_class)
#         if os.path.exists(multimodal_weights_path):
#             multimodal_model.load_weights(multimodal_weights_path)
#             y_pred_multimodal = np.argmax(multimodal_model.predict([X_test, probe_test]), axis=1)
#             accuracy_multimodal, precision_multimodal, recall_multimodal, f1_multimodal = calculate_metrics(y_test, y_pred_multimodal)
#             params_multimodal = multimodal_model.count_params()
#         else:
#             print(f"Weights for multimodal model not found at {multimodal_weights_path}")
#             continue
        
#         evaluation_data.append({
#             "model_name": model_name,
#             "baseline_params": params_baseline,
#             "baseline_accuracy": accuracy_baseline,
#             "baseline_precision": precision_baseline,
#             "baseline_recall": recall_baseline,
#             "baseline_f1": f1_baseline,
#             "multimodal_params": params_multimodal,
#             "multimodal_accuracy": accuracy_multimodal,
#             "multimodal_precision": precision_multimodal,
#             "multimodal_recall": recall_multimodal,
#             "multimodal_f1": f1_multimodal
#         })
    
#     return evaluation_data

In [ ]:
# evaluation_results = evaluate_trained_models(models, X_test, y_test, probe_test, results)

# # Print the evaluation results in the desired format
# import pandas as pd

# df_results = pd.DataFrame(evaluation_results)
# print(df_results)


In [ ]:
# df_results


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = np.mean(y_true == y_pred) * 100
    precision = precision_score(y_true, y_pred, average='weighted') * 100
    recall = recall_score(y_true, y_pred, average='weighted') * 100
    f1 = f1_score(y_true, y_pred, average='weighted') * 100
    return accuracy, precision, recall, f1

def evaluate_trained_models(models, X_test, y_test, probe_test):
    evaluation_data = []
    
    for model_name, model_class in models.items():
        # Define paths for model weights
        baseline_weights_path = f'./model_weights/{model_name}_baseline.weights.h5'
        multimodal_weights_path = f'./model_weights/{model_name}_multimodal.weights.h5'
        
        # Evaluate baseline model
        print(f'Evaluating {model_name} Baseline Model')
        baseline_model = build_baseline_model(model_class)
        if os.path.exists(baseline_weights_path):
            baseline_model.load_weights(baseline_weights_path)
            y_pred_baseline = np.argmax(baseline_model.predict(X_test), axis=1)
            accuracy_baseline, precision_baseline, recall_baseline, f1_baseline = calculate_metrics(y_test, y_pred_baseline)
            params_baseline = baseline_model.count_params()
            
            evaluation_data.append({
                "model_name": model_name,
                "model_type": "baseline",
                "params": params_baseline,
                "accuracy": accuracy_baseline,
                "precision": precision_baseline,
                "recall": recall_baseline,
                "f1": f1_baseline
            })
        else:
            print(f"Weights for baseline model not found at {baseline_weights_path}")
            continue
        
        # Evaluate multimodal model
        print(f'Evaluating {model_name} Multimodal Model')
        multimodal_model = build_multimodal_model(model_class)
        if os.path.exists(multimodal_weights_path):
            multimodal_model.load_weights(multimodal_weights_path)
            y_pred_multimodal = np.argmax(multimodal_model.predict([X_test, probe_test]), axis=1)
            accuracy_multimodal, precision_multimodal, recall_multimodal, f1_multimodal = calculate_metrics(y_test, y_pred_multimodal)
            params_multimodal = multimodal_model.count_params()
            
            evaluation_data.append({
                "model_name": model_name,
                "model_type": "multimodal",
                "params": params_multimodal,
                "accuracy": accuracy_multimodal,
                "precision": precision_multimodal,
                "recall": recall_multimodal,
                "f1": f1_multimodal
            })
        else:
            print(f"Weights for multimodal model not found at {multimodal_weights_path}")
            continue
    
    df_results = pd.DataFrame(evaluation_data)
    return df_results


In [ ]:
df_results = evaluate_trained_models(models, X_test, y_test, probe_test)

In [ ]:
df_results